## Content Based Recommendation System

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

### Data Loading

In [3]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("meeAtif/hadith_datasets")

C:\Users\nilan\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/meeAtif/hadith_datasets/resolve/cd62605e4fef3c46968a5f9fdcccbbbffdf631b5/hadith_datasets.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since meeAtif/hadith_datasets couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\nilan\.cache\huggingface\datasets\meeAtif___hadith_datasets\default\0.0.0\cd62605e4fef3c46968a5f9fdcccbbbffdf631b5 (last modified on Fri Jul 17 11:02:39 2026).


In [4]:
hadhith = pd.DataFrame(ds["train"])

In [5]:
hadhith.head()

,Book,Chapter_Number,Chapter_Title_Arabic,Chapter_Title_English,Arabic_Text,English_Text,Grade,Reference,In-book reference
0,Jami` at-Tirmidhi,1,باب مَا جَاءَ لاَ تُقْبَلُ صَلاَةٌ بِغَيْرِ طُ...,Chapter: What Has Been Related That Salat Is N...,حَدَّثَنَا قُتَيْبَةُ بْنُ سَعِيدٍ، حَدَّثَنَا...,"Ibn `Umar narrated that: the Prophet said: ""Sa...",Sahih (Darussalam),https://sunnah.com/tirmidhi:1,"Book 1, Hadith 1"
1,Jami` at-Tirmidhi,1,باب مَا جَاءَ لاَ تُقْبَلُ صَلاَةٌ بِغَيْرِ طُ...,Chapter: What Has Been Related That Salat Is N...,حَدَّثَنَا إِسْحَاقُ بْنُ مُوسَى الأَنْصَارِيّ...,Abu Hurairah narrated that : Allah's Messenger...,Sahih (Darussalam),https://sunnah.com/tirmidhi:2,"Book 1, Hadith 2"
2,Jami` at-Tirmidhi,1,باب مَا جَاءَ لاَ تُقْبَلُ صَلاَةٌ بِغَيْرِ طُ...,Chapter: What Has Been Related That Salat Is N...,حَدَّثَنَا قُتَيْبَةُ، وَهَنَّادٌ، وَمَحْمُودُ...,"'Ali narrated that : the Prophet, said: ""The k...",Hasan (Darussalam),https://sunnah.com/tirmidhi:3,"Book 1, Hadith 3"
3,Jami` at-Tirmidhi,1,باب مَا جَاءَ لاَ تُقْبَلُ صَلاَةٌ بِغَيْرِ طُ...,Chapter: What Has Been Related That Salat Is N...,حَدَّثَنَا أَبُو بَكْرٍ، مُحَمَّدُ بْنُ زَنْجَ...,"Jabir bin 'Abdullah, may Allah be pleased with...",Hasan (Darussalam),https://sunnah.com/tirmidhi:4,"Book 1, Hadith 4"
4,Jami` at-Tirmidhi,1,باب مَا جَاءَ لاَ تُقْبَلُ صَلاَةٌ بِغَيْرِ طُ...,Chapter: What Has Been Related That Salat Is N...,حَدَّثَنَا قُتَيْبَةُ، وَهَنَّادٌ، قَالاَ حَدّ...,"Anas bin Malik said: ""When the Prophet entered...",Sahih (Darussalam),https://sunnah.com/tirmidhi:5,"Book 1, Hadith 5"


### Preprocessing

In [6]:
hadhith.shape

(33738, 9)

In [7]:
hadhith.duplicated().sum()

np.int64(0)

In [8]:
hadhith.isna().sum()

Book                     0
Chapter_Number           0
Chapter_Title_Arabic     0
Chapter_Title_English    0
Arabic_Text              0
English_Text             0
Grade                    0
Reference                0
In-book reference        0
dtype: int64

In [9]:
hadhith.columns

Index(['Book', 'Chapter_Number', 'Chapter_Title_Arabic',
       'Chapter_Title_English', 'Arabic_Text', 'English_Text', 'Grade',
       'Reference', 'In-book reference'],
      dtype='object')

### TF-IDF

In [10]:
hadhith["combined"] =  hadhith["English_Text"]  + " " + hadhith["Arabic_Text"]

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(hadhith["combined"])

In [11]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1476113 stored elements and shape (33738, 26292)>

In [12]:
hadhith["display_title"] = (hadhith["Book"] + " " + hadhith["Chapter_Title_English"])
hadhith.head()

,Book,Chapter_Number,Chapter_Title_Arabic,Chapter_Title_English,Arabic_Text,English_Text,Grade,Reference,In-book reference,combined,display_title
0,Jami` at-Tirmidhi,1,باب مَا جَاءَ لاَ تُقْبَلُ صَلاَةٌ بِغَيْرِ طُ...,Chapter: What Has Been Related That Salat Is N...,حَدَّثَنَا قُتَيْبَةُ بْنُ سَعِيدٍ، حَدَّثَنَا...,"Ibn `Umar narrated that: the Prophet said: ""Sa...",Sahih (Darussalam),https://sunnah.com/tirmidhi:1,"Book 1, Hadith 1","Ibn `Umar narrated that: the Prophet said: ""Sa...",Jami` at-Tirmidhi Chapter: What Has Been Relat...
1,Jami` at-Tirmidhi,1,باب مَا جَاءَ لاَ تُقْبَلُ صَلاَةٌ بِغَيْرِ طُ...,Chapter: What Has Been Related That Salat Is N...,حَدَّثَنَا إِسْحَاقُ بْنُ مُوسَى الأَنْصَارِيّ...,Abu Hurairah narrated that : Allah's Messenger...,Sahih (Darussalam),https://sunnah.com/tirmidhi:2,"Book 1, Hadith 2",Abu Hurairah narrated that : Allah's Messenger...,Jami` at-Tirmidhi Chapter: What Has Been Relat...
2,Jami` at-Tirmidhi,1,باب مَا جَاءَ لاَ تُقْبَلُ صَلاَةٌ بِغَيْرِ طُ...,Chapter: What Has Been Related That Salat Is N...,حَدَّثَنَا قُتَيْبَةُ، وَهَنَّادٌ، وَمَحْمُودُ...,"'Ali narrated that : the Prophet, said: ""The k...",Hasan (Darussalam),https://sunnah.com/tirmidhi:3,"Book 1, Hadith 3","'Ali narrated that : the Prophet, said: ""The k...",Jami` at-Tirmidhi Chapter: What Has Been Relat...
3,Jami` at-Tirmidhi,1,باب مَا جَاءَ لاَ تُقْبَلُ صَلاَةٌ بِغَيْرِ طُ...,Chapter: What Has Been Related That Salat Is N...,حَدَّثَنَا أَبُو بَكْرٍ، مُحَمَّدُ بْنُ زَنْجَ...,"Jabir bin 'Abdullah, may Allah be pleased with...",Hasan (Darussalam),https://sunnah.com/tirmidhi:4,"Book 1, Hadith 4","Jabir bin 'Abdullah, may Allah be pleased with...",Jami` at-Tirmidhi Chapter: What Has Been Relat...
4,Jami` at-Tirmidhi,1,باب مَا جَاءَ لاَ تُقْبَلُ صَلاَةٌ بِغَيْرِ طُ...,Chapter: What Has Been Related That Salat Is N...,حَدَّثَنَا قُتَيْبَةُ، وَهَنَّادٌ، قَالاَ حَدّ...,"Anas bin Malik said: ""When the Prophet entered...",Sahih (Darussalam),https://sunnah.com/tirmidhi:5,"Book 1, Hadith 5","Anas bin Malik said: ""When the Prophet entered...",Jami` at-Tirmidhi Chapter: What Has Been Relat...


In [13]:
hadhith.shape

(33738, 11)

### Recommendation

In [14]:
def get_recommendation(user_input): 
    query_vector = tfidf.transform([user_input])
    cosine_sim1 = linear_kernel(query_vector, tfidf_matrix).flatten()
    similarity_scores = list(enumerate(cosine_sim1))
    similarity_scores = sorted(similarity_scores, key= lambda x: x[1], reverse = True)
    similarity_scores = similarity_scores[:10]
    similarity_index = [i[0] for i in similarity_scores]
    
    print(hadhith["combined"].iloc[similarity_index])

get_recommendation("patience")

25303    It was\nnarrated from Anas bin Malik that the ...
25448    It was\nnarrated from Abu Mujibah Al-Bahili th...
29926    It was narrated that Thabit said: "I heard Ana...
12680    Narrated Anas: The Prophet (ﷺ) said, "The real...
988      Anas narrated that: The Messenger of Allah sai...
30465    It was narrated that Abu Hurairah said: " I he...
17651    Narrated Abu Sa`id: Some people from the Ansar...
989      Anas bin Malik narrated that: The Messenger of...
31213    It was narrated that Abu Hurairah said: "A man...
8708     It has been narrated on the authority of Ibn '...
Name: combined, dtype: object


### Saving Files

In [15]:
import pickle

In [16]:
with open("tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)
    
with open("hadhith.pkl", "wb") as f:
    pickle.dump(hadhith, f)